In [ ]:
import pandas as pd
import re
import itertools
import json
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import re

In [48]:
tour_df = pd.read_csv("../data/taylor_swift_tour_df_labeled.csv", parse_dates=["event_date"])
tour_df.head()

,event_date,venue,city,country,tour,set_name,song_name,is_cover,info,supporting_album,tour_type,is_new_album_song
0,2024-12-08,BC Place Stadium,Vancouver,Canada,The Eras Tour,main,NaN,False,"w/ elements of MA&tHP, The Alchemy, Fearless, ...",Midnights,retrospective,False
1,2024-12-08,BC Place Stadium,Vancouver,Canada,The Eras Tour,Lover,Miss Americana & the Heartbreak Prince,False,shortened,Midnights,retrospective,False
2,2024-12-08,BC Place Stadium,Vancouver,Canada,The Eras Tour,Lover,Cruel Summer,False,extended outro,Midnights,retrospective,False
3,2024-12-08,BC Place Stadium,Vancouver,Canada,The Eras Tour,Lover,The Man,False,spoken intro,Midnights,retrospective,False
4,2024-12-08,BC Place Stadium,Vancouver,Canada,The Eras Tour,Lover,You Need to Calm Down,False,shortened,Midnights,retrospective,False


In [49]:
tour_order = [
    "Fearless", "Speak Now World Tour", "The Red Tour",
    "The 1989 World Tour", "reputation Stadium Tour", "The Eras Tour"
]
tour_index = {tour: i for i, tour in enumerate(tour_order)}
tour_df["tour_index"] = tour_df["tour"].map(tour_index)

In [50]:
tour_song_df = tour_df.groupby(["tour", "tour_index", "song_name"]).agg(
    num_shows_played=("event_date", "nunique"),
    supporting_album=("supporting_album", "first"),
    tour_type=("tour_type", "first"),
    is_new_album_song=("is_new_album_song", "first"),
).reset_index()

print(tour_song_df.shape)
tour_song_df.head()

(673, 7)


,tour,tour_index,song_name,num_shows_played,supporting_album,tour_type,is_new_album_song
0,Fearless,0,Change,47,Fearless,single_album,True
1,Fearless,0,Fearless,110,Fearless,single_album,True
2,Fearless,0,Fifteen,108,Fearless,single_album,True
3,Fearless,0,Forever & Always,110,Fearless,single_album,True
4,Fearless,0,Hey Stephen,106,Fearless,single_album,True


In [51]:

#build df of whether song was played at each show
all_songs = tour_df["song_name"].unique()

full_grid = pd.DataFrame(
    itertools.product(tour_order, all_songs),
    columns=["tour", "song_name"]
)
full_grid["tour_index"] = full_grid["tour"].map(tour_index)

full_grid = full_grid.merge(
    tour_song_df[["tour", "song_name", "num_shows_played"]],
    on=["tour", "song_name"],
    how="left"
)
full_grid["was_played_on_tour"] = full_grid["num_shows_played"].notna().astype(int)
full_grid["num_shows_played"] = full_grid["num_shows_played"].fillna(0)

print(full_grid.shape)
full_grid["was_played_on_tour"].value_counts()

(2940, 5)


was_played_on_tour
0    2267
1     673
Name: count, dtype: int64

In [54]:

song_origin_album = tour_song_df.sort_values("tour_index").groupby("song_name")["tour_index"].min()

full_grid["song_origin_tour_index"] = full_grid["song_name"].map(song_origin_album)

# only keep rows where the song could have plausibly existed by this tour
full_grid = full_grid[full_grid["song_origin_tour_index"] <= full_grid["tour_index"]].copy()

print(full_grid.shape)
full_grid.groupby("tour")["was_played_on_tour"].mean()

(1255, 6)


tour
Fearless                   1.000000
Speak Now World Tour       0.823009
The 1989 World Tour        0.381818
The Eras Tour              0.662577
The Red Tour               0.454545
reputation Stadium Tour    0.303150
Name: was_played_on_tour, dtype: float64

In [ ]:
#save reengineered dat
#full_grid.to_csv("../data/taylor_swift_full_grid.csv", index=False)

In [ ]:
#feature 1: whether song was played in the preceding tour
full_grid = full_grid.sort_values(["song_name", "tour_index"])
full_grid["played_last_tour"] = full_grid.groupby("song_name")["was_played_on_tour"].shift(1)
full_grid["played_last_tour"] = full_grid["played_last_tour"].fillna(0)

In [ ]:
#feature 2: how many (eligible) tours was the song a part of
# full_grid["cumulative_play_rate"] = (
#     full_grid.groupby("song_name")["was_played_on_tour"]
#     .apply(lambda x: x.shift(1).expanding().mean())
#     .reset_index(level=0, drop=True)
# )
# full_grid["cumulative_play_rate"] = full_grid["cumulative_play_rate"].fillna(0)

In [ ]:
#feature 3: how many tours old the song is
full_grid["song_age_tours"] = full_grid["tour_index"] - full_grid["song_origin_tour_index"]

In [ ]:
#feature 4: is the song from a current album the tour is promoting
# full_grid = full_grid.merge(
#     tour_song_df[["tour", "song_name", "is_new_album_song"]],
#     on=["tour", "song_name"],
#     how="left"
# )
# full_grid["is_new_album_song"] = full_grid["is_new_album_song"].fillna(False)

In [ ]:
#feature 5: single album tour or career-spanning tour
tour_type = {
    "Fearless": "single_album", "Speak Now World Tour": "single_album",
    "The Red Tour": "single_album", "The 1989 World Tour": "single_album",
    "reputation Stadium Tour": "single_album", "The Eras Tour": "retrospective"
}
full_grid["tour_type"] = full_grid["tour"].map(tour_type)
full_grid_encoded = pd.get_dummies(full_grid, columns=["tour_type"], drop_first=True)

In [ ]:
# full_grid.to_csv("../data/taylor_swift_model_df.csv", index=False)
# print(full_grid.shape)
# full_grid.head()

(1255, 11)


,tour,song_name,tour_index,num_shows_played,was_played_on_tour,song_origin_tour_index,played_last_tour,cumulative_play_rate,song_age_tours,is_new_album_song,tour_type
0,The Eras Tour,"""Slut!""",5,1.0,1,5.0,0.0,0.0,0.0,False,retrospective
1,The Eras Tour,"""Slut!"" / False God",5,1.0,1,5.0,0.0,0.0,0.0,False,retrospective
2,The Red Tour,"""The Lucky One""",2,86.0,1,2.0,0.0,0.0,0.0,False,single_album
3,The 1989 World Tour,"""The Lucky One""",3,0.0,0,2.0,1.0,1.0,1.0,False,single_album
4,reputation Stadium Tour,"""The Lucky One""",4,0.0,0,2.0,0.0,0.5,2.0,False,single_album


In [ ]:
with open("../data/taylor_swift_lyrics.json") as f:
    lyrics_data = json.load(f)

'''
uses VADER, a sentiment scoring tool
it is used to score the mood of the song
output: score from -1 to 1
    -1 represents negative (melancholy) lyrics
    1 represents positive (upbeat)lyrics
'''
analyzer = SentimentIntensityAnalyzer()


In [ ]:
def compute_lyric_features(lyrics):
    '''
    this function computes 3 features
    1. average sentence length 
        purpose: to characterise a song as one made of short, punchy phrases
        or longer flowing sentences like a narrative
        output: a single number (float) per song
    2. proper noun density
        purpose: how specific a song's lyrics are, 
            especially important since Swift uses many references such as "Dear John"
        output: a single number (small float) per song
    3. unique word ratio
        purpose: how repetitive a song's lyrics are
        output: a decimal between 0 and 1 (ratio closer to 1 means more uniqueness)
    '''
    if not lyrics:
        return None, None, None
    words = lyrics.split()
    sentences = re.split(r'[.!?]', lyrics)
    avg_sentence_len = len(words) / max(len(sentences), 1)
    proper_noun_count = len(re.findall(r'(?<!\. )(?<!^)[A-Z][a-z]+', lyrics))
    proper_noun_density = proper_noun_count / max(len(words), 1)
    unique_word_ratio = len(set(w.lower() for w in words)) / max(len(words), 1)
    return avg_sentence_len, proper_noun_density, unique_word_ratio

In [ ]:
lyric_rows = []
for song, lyrics in lyrics_data.items():
    sentiment = analyzer.polarity_scores(lyrics)["compound"] if lyrics else None
    avg_len, noun_density, word_ratio = compute_lyric_features(lyrics)
    lyric_rows.append({
        "song_name": song, "sentiment": sentiment,
        "avg_sentence_len": avg_len, "proper_noun_density": noun_density,
        "unique_word_ratio": word_ratio
    })

lyric_features_df = pd.DataFrame(lyric_rows)
lyric_features_df.to_csv("../data/taylor_swift_lyric_features.csv", index=False)

full_grid_encoded = full_grid_encoded.merge(lyric_features_df, on="song_name", how="left")
for col in ["sentiment", "avg_sentence_len", "proper_noun_density", "unique_word_ratio"]:
    full_grid_encoded[col] = full_grid_encoded[col].fillna(0)

In [ ]:
full_grid_encoded.to_csv("../data/taylor_swift_model_df_final_nlp.csv", index=False)